# Heuristic Intervention

In this example, we will demonstrate the use of a heuristic intervention.
These interventions are not based on rigorous forecasts or formal optimization,
but instead are just heuristic tweaks that try to improve revenue by changing
parts of the RM system. For example, the heuristic we demonstrate here is a 
booked load factor curve adjustment, which takes an exogenously provided booked
load factor curve for individual legs, and nudges the mean forecast up or down
if the leg's actual bookings are running much higher or lower than expected at
any point in the booking curve. This is meant to replicate what a human analyst
might do in a similar situation.

In [ ]:
import pandas as pd

import passengersim as pax

pax.versions()

We'll start by loading the standard 3MKT demo model.

In [ ]:
cfg = pax.Config.from_yaml(pax.demo_network("3MKT/DEMO"))

The heuristic adjustment is made using a `RmAction`, which works just like any other `RmAction`
when it is inserted in a `RmSys`'s action list.

In [ ]:
from passengersim.rm.emsr import ExpectedMarginalSeatRevenue
from passengersim.rm.heuristic_adjustments.booked_lf import BookedLoadFactorAdjustment
from passengersim.rm.standard_forecasting import StandardLegForecast
from passengersim.rm.systems import RmSys, RmSysOption, register_rm_system
from passengersim.rm.untruncation import LegUntruncation


@register_rm_system
class EX(RmSys):
    availability_control = "leg"
    """This RM system uses leg-level class allocation availability controls."""

    actions = [
        LegUntruncation,
        StandardLegForecast,
        BookedLoadFactorAdjustment,
        ExpectedMarginalSeatRevenue,
    ]

The `BookedLoadFactorAdjustment` uses a set of booked load factor curves, which are stored
in a configuration's `other_controls` attribute. This attribute allows users to set arbitrary
other controls, which can be loaded, validated, and used by `RmAction` steps that call them.
For the booked load factor adjustment, the needed inputs are stored under the 
"booked_load_factor_curves" key.  These curves are already included in the demo configuration,
so we don't need to do anything special to make them available.

In [ ]:
cfg.other_controls["booked_load_factor_curves"]

You may not that this configuration is itself a dictionary, which has a couple of different curves
stored under different keys:

In [ ]:
cfg.other_controls["booked_load_factor_curves"].keys()

To tell the action which curve to use for which leg, we'll attach a 'BLF_Curve' tag to each leg,
which tells the simulation which curve to use.  For this example, we'll attach the "BOSTON"
curve to the legs originating at BOS, and the "CHICAGO" tag to legs originating at ORD.  This is
our choice in assigning these tags; we could alternatively assign tags based on stage length, 
departure time of day, or any other leg attribute(s) we like.  In the extreme, we could create and
assign a unique curve to every individual leg in the entire simulation.

In [ ]:
for leg in cfg.legs:
    if leg.carrier == "AL1":
        if leg.orig == "BOS":
            leg.tags["BLF_Curve"] = "BOSTON"
        if leg.orig == "ORD":
            leg.tags["BLF_Curve"] = "CHICAGO"

We're now pretty much all set up and use this new RM system in a simulation in place of a standard "E" type system.

In [ ]:
cfg.carriers.AL1.rm_system = "EX"

Let's run it and see what happens.

In [ ]:
sim = pax.MultiSimulation(cfg)

In [ ]:
summary = sim.run()

In [ ]:
summary.fig_carrier_revenues()

In [ ]:
summary.fig_carrier_load_factors()

In [ ]:
summary.fig_fare_class_mix()

In [ ]:
summary.fig_carrier_head_to_head_revenue("AL1", "AL2")